In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import pandas as pd
import lazyqsar
import joblib
import numpy as np
import os

st = "alphafold2_P9WFS9_model_0_pocket_1"
perc = "bin_1"

root = '.'
# root = os.path.dirname(os.path.abspath(__file__))

# Get input data
PATH_TO_REPORTS = os.path.join(root, "..", "processed", "unidock_docking", "binarized_reports")
PATH_TO_OUTPUT = os.path.join(root, "..", "processed", "unidock_docking", "models", st, perc)
PATH_TO_EMBEDDINGS = os.path.join(root, "..", "processed", "enamine_characterization")
os.makedirs(PATH_TO_OUTPUT, exist_ok=True)

# Load compounds and activities
report = pd.read_csv(os.path.join(PATH_TO_REPORTS, f"report_bin_{st}.csv"))
compounds = report["compound"].tolist()
Y = np.array(report[perc].tolist())

# Load ids and embeddings
ids = open(os.path.join(PATH_TO_EMBEDDINGS, "IDs_CheMeleon.txt")).read().splitlines()
embeddings = np.load(os.path.join(PATH_TO_EMBEDDINGS, "X_CheMeleon.npz"))['X']

# Mapping id to embedding
id_to_embedding = {i: j for i,j in zip(ids, embeddings)}

# Creating matrix
X = np.array([id_to_embedding[i] for i in compounds])

# Stratified 5-fold CV
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []

In [ ]:
import random
random.seed(42)
sample_indices = random.sample(range(0, 100000), 5000)
X, Y = X[sample_indices], Y[sample_indices]

In [ ]:
for train_idx, test_idx in kf.split(X, Y):

    # Train test split
    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]

    # Train model only on training set
    model = lazyqsar.LazyBinaryClassifier(model_type="random_forest", pca=False, min_seen_across_partitions=1, 
                                          num_trials=20, base_num_splits=1, max_samples=10000)
    model.fit(X_train, Y_train)
    probs = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(Y_test, probs)
    aucs.append(auc)

Total samples: 80123, positive samples: 801, negative samples: 79322
Maximum samples per partition: 10000, minimum samples per partition: 30
Positive proportion: 0.01
Original positive samples: 801, total samples: 80123
Maximum samples: 10000
Sampling 801 positive and 9199 negative samples from 10000 total samples.


  2%|▎         | 25/1000 [00:01<01:11, 13.68it/s]


All indices seen at least 1 times. Stopping sampling.
Unique sampled indices matrix shape: (26, 10000)


100%|██████████| 26/26 [01:06<00:00,  2.57s/it]


Indices matrix shape after redundancy removal: (9, 10000)
Original positive negative balance: positive 801, negative 79322
Avg positive samples: 801.0, avg negative samples: 9199.0


INFO:flaml.default.suggest:metafeature distance: 2.3496384419909924
[I 2025-07-24 17:40:10,279] A new study created in memory with name: no-name-4c90e9cc-f541-4b99-a41b-7206b942201c


Fitting model on 10000 samples, positive samples: 801, negative samples: 9199, number of features 1941
Suggested zero-shot hyperparameters: {'n_estimators': 501, 'max_features': 0.24484242524861066, 'criterion': 'entropy', 'max_leaf_nodes': 1156, 'random_state': 12032022, 'verbose': 0, 'class_weight': 'balanced_subsample'}
Fitting...


[W 2025-07-24 17:40:50,714] Trial 0 failed with parameters: {'n_estimators': 501, 'max_features': 0.24484242524861066, 'max_leaf_nodes': 1156, 'criterion': 'entropy'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/acomajuncosa/miniconda3/envs/lazyqsar/lib/python3.11/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/home/acomajuncosa/Documents/lazy-qsar/lazyqsar/models/random_forest_binary_classifier.py", line 204, in objective_with_custom_early_stop
    score = self._objective(trial, X, y, hyperparam_search)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/acomajuncosa/Documents/lazy-qsar/lazyqsar/models/random_forest_binary_classifier.py", line 150, in _objective
    model.fit(train_x, train_y)
  File "/home/acomajuncosa/miniconda3/envs/lazyqsar/lib/python3.11/site-packages/sklearn/base.py", line 1389, in wrapper
 

KeyboardInterrupt: 

In [ ]:
with open(os.path.join(PATH_TO_OUTPUT, f"AUROCs.csv"), "w") as f:
    f.write(",".join([str(round(i, 3)) for i in aucs]))

In [ ]:
# Train model only on all data
model = lazyqsar.LazyBinaryClassifier(model_type="random_forest", pca=False, min_seen_across_partitions=1, 
                                        num_trials=20, base_num_splits=1, max_samples=10000)
model.fit(X, Y)

In [ ]:
# Save model
joblib.dump(model, os.path.join(PATH_TO_OUTPUT, "LQ_RF.joblib"))